# SIPTA Notebook: Validación de datos — Salud

## Dominio: Salud

Este notebook valida la calidad estructural del dataset
**Instituciones de Salud con servicios de urgencias en Bogotá D.C.**

### Objetivos

- Verificar dimensiones y esquema del dataset.
- Identificar valores nulos.
- Detectar duplicados exactos.
- Evaluar la unicidad de las sedes.
- Validar la disponibilidad y calidad de las coordenadas geográficas.
- Determinar si el dataset es apto para una futura asociación con la unidad territorial `Localidad`.

> Esta etapa no realiza limpieza, integración territorial ni cálculo de indicadores.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SALUD_DIR = PROJECT_ROOT / "data" / "raw" / "SALUD"

archivo_salud = SALUD_DIR / "osb_ofertasrv-ips-urgencias.csv"

df_salud = pd.read_csv(
    archivo_salud,
    sep=None,
    engine="python",
    encoding="cp1252"
)

print("Shape:", df_salud.shape)

Shape: (84, 11)


In [2]:
print("=== VALORES NULOS POR COLUMNA ===")

nulos = pd.DataFrame({
    "nulos": df_salud.isna().sum(),
    "porcentaje": (df_salud.isna().mean() * 100).round(2)
})

display(nulos)

=== VALORES NULOS POR COLUMNA ===


,nulos,porcentaje
OBJECTID,0,0.00
Código IPS,0,0.00
Nombre IPS,0,0.00
Nombre sede,0,0.00
Número sede,0,0.00
Dirección,0,0.00
Telefono contacto,1,1.19
Correco electrónico,0,0.00
Tipo de naturaleza,0,0.00
Latitud,0,0.00


In [3]:
print("=== DUPLICADOS EXACTOS ===")

duplicados_exactos = df_salud.duplicated().sum()
porcentaje = duplicados_exactos / len(df_salud) * 100

print("Filas duplicadas exactas:", duplicados_exactos)
print(f"Porcentaje: {porcentaje:.4f}%")

=== DUPLICADOS EXACTOS ===
Filas duplicadas exactas: 0
Porcentaje: 0.0000%


In [4]:
print("=== UNICIDAD DE SEDE ===")

sedes_unicas = (
    df_salud[["Código IPS", "Número sede"]]
    .drop_duplicates()
    .shape[0]
)

print("Registros totales:", len(df_salud))
print("Combinaciones únicas Código IPS + Número sede:", sedes_unicas)

duplicados_sede = df_salud.duplicated(
    subset=["Código IPS", "Número sede"]
).sum()

print("Duplicados según clave de sede:", duplicados_sede)

=== UNICIDAD DE SEDE ===
Registros totales: 84
Combinaciones únicas Código IPS + Número sede: 84
Duplicados según clave de sede: 0


In [5]:
print("=== IPS Y SEDES ===")

print("IPS diferentes:", df_salud["Código IPS"].nunique())
print("Sedes diferentes:", sedes_unicas)

=== IPS Y SEDES ===
IPS diferentes: 44
Sedes diferentes: 84


In [6]:
print("=== CONVERSIÓN DE COORDENADAS ===")

latitud_num = pd.to_numeric(
    df_salud["Latitud"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

longitud_num = pd.to_numeric(
    df_salud["Longitud"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Latitudes no convertibles:", latitud_num.isna().sum())
print("Longitudes no convertibles:", longitud_num.isna().sum())

print("\n=== LATITUD NUMÉRICA ===")
print(latitud_num.describe())

print("\n=== LONGITUD NUMÉRICA ===")
print(longitud_num.describe())

=== CONVERSIÓN DE COORDENADAS ===
Latitudes no convertibles: 0
Longitudes no convertibles: 0

=== LATITUD NUMÉRICA ===
count    84.000000
mean      4.634528
std       0.105497
min       4.029030
25%       4.592257
50%       4.636102
75%       4.692857
max       4.760767
Name: Latitud, dtype: float64

=== LONGITUD NUMÉRICA ===
count    84.000000
mean    -74.094681
std       0.044066
min     -74.315131
25%     -74.120774
50%     -74.088897
75%     -74.064930
max     -74.023025
Name: Longitud, dtype: float64


In [7]:
print("=== VALIDEZ GEOGRÁFICA BÁSICA ===")

latitud_invalida = ~latitud_num.between(-90, 90)
longitud_invalida = ~longitud_num.between(-180, 180)

print("Latitudes fuera del rango válido:", latitud_invalida.sum())
print("Longitudes fuera del rango válido:", longitud_invalida.sum())

coordenadas_invalidas = latitud_invalida | longitud_invalida

print(
    "Registros con alguna coordenada geográficamente inválida:",
    coordenadas_invalidas.sum()
)

=== VALIDEZ GEOGRÁFICA BÁSICA ===
Latitudes fuera del rango válido: 0
Longitudes fuera del rango válido: 0
Registros con alguna coordenada geográficamente inválida: 0


In [8]:
print("=== UNICIDAD DE COORDENADAS ===")

coordenadas = pd.DataFrame({
    "latitud": latitud_num,
    "longitud": longitud_num
})

print("Registros:", len(coordenadas))
print(
    "Pares de coordenadas distintos:",
    coordenadas.drop_duplicates().shape[0]
)

print(
    "Pares de coordenadas repetidos:",
    coordenadas.duplicated().sum()
)

=== UNICIDAD DE COORDENADAS ===
Registros: 84
Pares de coordenadas distintos: 84
Pares de coordenadas repetidos: 0


## Notas de validación — Salud

- El dataset contiene **84 registros y 11 variables**.
- La granularidad observada corresponde a **sedes de Instituciones Prestadoras de Servicios de Salud (IPS) con servicios de urgencias**.
- Se identifican **44 IPS diferentes y 84 sedes**.
- La combinación `Código IPS + Número sede` identifica de forma única los 84 registros:
  - combinaciones únicas: **84**;
  - duplicados según esta clave: **0**.
- No se encontraron duplicados exactos.
- Se identificó **1 valor nulo en `Telefono contacto`**, equivalente al **1,19 %** de los registros.
- Las variables relevantes para la identificación y territorialización (`Código IPS`, `Número sede`, `Latitud` y `Longitud`) no presentan valores nulos.
- Las columnas `Latitud` y `Longitud` se encuentran originalmente almacenadas como texto con coma decimal.
- La conversión temporal de las coordenadas a valores numéricos fue exitosa para los **84 registros**, sin valores no convertibles.
- Los valores observados se encuentran en los rangos:
  - latitud: aproximadamente **4.029 a 4.761**;
  - longitud: aproximadamente **-74.315 a -74.023**.
- No se encontraron valores fuera de los rangos geográficos universales válidos (`[-90, 90]` para latitud y `[-180, 180]` para longitud).
- Los **84 registros presentan pares de coordenadas distintos**.
- El dataset **no contiene la variable `Localidad` de manera explícita**.
- Sin embargo, la disponibilidad completa y consistente de coordenadas permite considerar el dataset **técnicamente apto para una futura asociación territorial con `Localidad`**.
- La asignación espacial efectiva de cada sede a una localidad se realizará posteriormente en la fase de **integración territorial**, utilizando la geometría oficial correspondiente.

### Decisión de validación

**Estado: APTO PARA CONTINUAR EL PIPELINE.**

El dataset presenta una calidad estructural adecuada para continuar hacia las siguientes etapas.  
El valor nulo identificado en `Telefono contacto` no afecta la identificación de las sedes ni la futura territorialización.

La conversión y estandarización definitiva de las coordenadas no se realiza sobre el archivo `raw`; deberá efectuarse en la etapa correspondiente de limpieza/estandarización.